In [1]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_curve, auc)
OUT = "outputs_fraud"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(42)

In [2]:
class Gaussian_Anomly_Detector:
    def fit(self,x):
        self.mu=x.mean(axis=0)
        self.var=x.var(axis=0)
        return self
    def log_prob(self,x):#يحسب لوغاريتم احتمال كل صف. كلما كانت النتيجة أقل، كان الصف أبعد عن النمط المعتاد.
        return(-0.5*np.log(2*np.pi * self.var)-(x-self.mu)**2/(2*self.var)).sum(axis=1)
    def predicte(self,x,log_eps):#قارن النتيجة بعتبة log_eps
        return (self.log_prob(x)<log_eps).astype(int)

In [14]:
def select_threshold(log_p,y):
    dest_eps,best_f1=None,-1#تبدأ بأفضل نتيجة 
    for eps in np.linspace(log_p.min(),np.percentile(log_p,99.5),1000):
        pred=(log_p<eps).astype(int)
        f1=f1_score(y,pred,zero_division=0)
        if f1 > best_f1:
            best_eps,best_f1=eps,f1
    return best_eps,best_f1        

In [15]:
def load_data(demo=False,path="creditcard.csv"):
    if demo or not os.path.exists(path):
        print(" Using synthetic demo data not found")
        n,nf=20000,28
        x=rng.normal(0,1,(n,nf))
        y=np.zero(n,dtype=int)
        idx=rng.choice(n,60,replace=False)
        x[idx]+=rng.normal(0,3,(60,nf))+2
        y[idx]=1
        df = pd.DataFrame(X, columns=[f"V{i}" for i in range(1, nf + 1)])
        df["Time"] = np.arange(n)
        df["Amount"] = np.abs(rng.lognormal(3, 1.2, n))
        df["Class"] = y
        return df
    return pd.read_csv(path)    
    

In [18]:
def main(demo):
    df=load_data(demo) #if true will read from demo else will read from file
    print(f"Shape:{df.shape}")
    print(df['Class'].value_counts().rename({0:"Normal",1:"Fraud"}))
    print(f"Fraud rate: {df["Class"].mean():0.4}\n")#calculate avg for Class column and display it as hundred perecentage with four digit 
    df["LogAmount"]=np.log1p(df["Amount"])#قد يحتوي على مبالغ صغيرة كثيرة وقيم كبيرة قليلة، فيكون توزيعه مائلًا. التحويل اللوغاريتمي يضغط القيم الكبيرة ويجعل نطاقها أقرب لبعضه، وقد يساعد النموذج على التعامل معها. استخدام
    features=[c for c in df.columns if c.startswith("V")]+["LogAmount"] # create feature list from v1 to v28 and LogAmount
    x_all,y_all=df[features].values,df["Class"].values #convert data to two numpy array x_all = feature value ,y_all=class
    normal=np.where(y_all==0)[0];fraud=np.where(y_all==1)[0]
    rng.shuffle(normal); rng.shuffle(fraud) # يخلط ترتيب المواقع في كل قائمة عشوائيًا، حتى لا يعتمد التقسيم على ترتيب البيانات الأصلي.
    n_tr = int(0.6 * len(normal)); n_cv = int(0.2 * len(normal))
    tr_idx = normal[:n_tr]#يختار أول 60% من مواقع العينات الطبيعية بعد خلطها. هذه ستكون بيانات التدريب، ولذلك لا تحتوي على حالات احتيال.
    cv_idx = np.r_[normal[n_tr:n_tr + n_cv], fraud[: len(fraud) // 2]]# merge to parts %20 from normal data and half fraud data
    te_idx = np.r_[normal[n_tr + n_cv:], fraud[len(fraud) // 2:]]
    x_tr=x_all[tr_idx]
    x_cv,y_cv=x_all[cv_idx],y_all[cv_idx]
    x_te,y_te=x_all[te_idx],y_all[te_idx]
    print(f"Train: {len(x_tr)} (all normal) | CV: {len(x_cv)} ({y_cv.sum()} fraud) "
          f"| Test: {len(x_te)} ({y_te.sum()} fraud)\n")
    '''- عدد عينات التدريب، مع توضيح أنها كلها طبيعية.
- عدد عينات CV وعدد حالات الاحتيال فيها.
- عدد عينات الاختبار وعدد حالات الاحتيال فيها.
بما أن قيم y_cv وy_te هي صفر أو واحد، فإن sum() يساوي عدد حالات الاحتيال في المجموعة.'''
    
    det=Gaussian_Anomly_Detector().fit(x_tr)
    eps,best_f1=select_threshold(det.log_prob(x_cv),y_cv)
    print(f"Best log(epsilon) = {eps:.2f}  |  CV F1 = {best_f1:.3f}\n")
    
    
    pred=det.predicte(x_te,eps)
    p=precision_score(y_te,pred,zero_division=0)
    r=recall_score(y_te,pred,zero_division=0)
    f1=f1_score(y_te,pred,zero_division=0)
    cm=confusion_matrix(y_te,pred)
    print("=== TEST RESULTS ===")
    print(f"Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f}")
    print("Confusion matrix [[TN FP],[FN TP]]:\n", cm, "\n")
    print("NOTE: accuracy is NOT used - predicting 'all normal' gives ~99.8% accuracy!\n")
    
    
    
    lp_te = det.log_prob(x_te)
    plt.figure(figsize=(8, 4))
    lo = np.percentile(lp_te, 0.1)
    plt.hist(lp_te[y_te == 0], bins=80, alpha=0.6, label="Normal", density=True, range=(lo, lp_te.max()))
    plt.hist(lp_te[y_te == 1], bins=40, alpha=0.7, label="Fraud", density=True, range=(lo, lp_te.max()))
    plt.axvline(eps, color="k", ls="--", label="threshold")
    plt.xlabel("log p(x)"); plt.legend(); plt.title("Score distribution (test)")
    plt.tight_layout(); plt.savefig(f"{OUT}/score_distribution.png", dpi=150); plt.close()

    prec, rec, _ = precision_recall_curve(y_te, -lp_te)
    plt.figure(figsize=(5, 4))
    plt.plot(rec, prec); plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"PR curve (AUC = {auc(rec, prec):.3f})")
    plt.tight_layout(); plt.savefig(f"{OUT}/pr_curve.png", dpi=150); plt.close()
    
    order = np.argsort(lp_te)  # most suspicious first
    for n in (100, 500):
        caught = y_te[order[:n]].sum()
        print(f"Review top {n} suspicious transactions -> catch {caught}/{y_te.sum()} frauds")
    print(f"\nDone. Results saved in ./{OUT}/")

In [19]:
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true")
    args, _ = ap.parse_known_args()
    main(args.demo)

Shape:(284807, 31)
Class
Normal    284315
Fraud        492
Name: count, dtype: int64
Fraud rate: 0.001727

Train: 170589 (all normal) | CV: 57109 (246 fraud) | Test: 57109 (246 fraud)

Best log(epsilon) = -254.97  |  CV F1 = 0.372

=== TEST RESULTS ===
Precision: 0.402 | Recall: 0.382 | F1: 0.392
Confusion matrix [[TN FP],[FN TP]]:
 [[56723   140]
 [  152    94]] 

NOTE: accuracy is NOT used - predicting 'all normal' gives ~99.8% accuracy!

Review top 100 suspicious transactions -> catch 53/246 frauds
Review top 500 suspicious transactions -> catch 144/246 frauds

Done. Results saved in ./outputs_fraud/


Shape:(284807, 31)


AttributeError: 'list' object has no attribute 'value_counts'